# Catastrophe Exposure Model — Multi-Peril Portfolio Roll-Up

End-to-end cat-exposure workflow on Wherobots. For a portfolio of insured
properties, score each location against three perils (flood, wildfire,
seismic), aggregate exposure by peril × region, and land the result in a
managed Iceberg table for downstream BI, reinsurance submissions, or
accumulation-control queries.

## Hazard sources

| Peril | Source | Representation |
|---|---|---|
| **Flood** | `wherobots_open_data.partner_samples.regrid_parcels` | FEMA flood zone tagged on each parcel polygon; point-in-polygon tags portfolio locations |
| **Wildfire** | `wherobots_pro_data.weather.wild_fires` (FPA FOD, 1992-2015) | Fire ignition points, buffered by `sqrt(FIRE_SIZE_acres / π)` to approximate burn footprint |
| **Seismic** | Inline PGA zones (USGS-derived bands for SF Bay, LA Basin, Cascadia, New Madrid, Wasatch Front) | Static hazard polygons declared inline |

## Output

`org_catalog.catastrophe_demo.portfolio_exposure` — one row per
(policy × peril) with `hazard_score`, `exposed_tiv`, and geometry, written
as Iceberg via `CREATE OR REPLACE TABLE`.

## 1. Setup

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

TARGET_DATABASE = "catastrophe_demo"
TARGET_TABLE    = "portfolio_exposure"
TARGET_FQN      = f"org_catalog.{TARGET_DATABASE}.{TARGET_TABLE}"

## 2. Portfolio Ingestion

12 insured properties across the U.S. West and central seismic zones,
chosen so each of the three perils is meaningfully represented. In
production, `portfolio_df` would read from the carrier's policy master
(Parquet on S3, JDBC to Guidewire, etc.).

In [ ]:
portfolio = [
    # policy_id, property_name,           lon,       lat,     tiv,     state, occupancy
    ("POL-001", "SF Marina Tower",        -122.4345, 37.8020, 8500000.0, "CA", "Commercial"),
    ("POL-002", "Napa Vineyard Estate",   -122.2869, 38.5025, 4200000.0, "CA", "Residential"),
    ("POL-003", "Lake County Vineyard",   -122.7700, 38.8500, 3100000.0, "CA", "Commercial"),
    ("POL-004", "Newport Balboa",         -117.9022, 33.6019, 3400000.0, "CA", "Residential"),
    ("POL-005", "LA Downtown Office",     -118.2580, 34.0430, 9800000.0, "CA", "Commercial"),
    ("POL-006", "Sequoia Foothill Lodge", -118.9100, 36.8800, 1400000.0, "CA", "Commercial"),
    ("POL-007", "Sierra Foothill Ranch",  -120.7000, 38.3500,  980000.0, "CA", "Residential"),
    ("POL-008", "Portland Riverfront",    -122.6784, 45.5152, 3100000.0, "OR", "Commercial"),
    ("POL-009", "Seattle Bellevue",       -122.2015, 47.6101, 5500000.0, "WA", "Residential"),
    ("POL-010", "Tacoma Warehouse",       -122.4443, 47.2529, 2250000.0, "WA", "Industrial"),
    ("POL-011", "Memphis Distribution",    -90.0490, 35.1495, 4800000.0, "TN", "Industrial"),
    ("POL-012", "Salt Lake Tech Park",    -111.8910, 40.7608, 3600000.0, "UT", "Commercial"),
]

portfolio_schema = StructType([
    StructField("policy_id",     StringType()),
    StructField("property_name", StringType()),
    StructField("lon",           DoubleType()),
    StructField("lat",           DoubleType()),
    StructField("tiv",           DoubleType()),
    StructField("state",         StringType()),
    StructField("occupancy",     StringType()),
])

portfolio_df = sedona.createDataFrame(portfolio, portfolio_schema) \
    .withColumn("geometry",
                f.expr("ST_SetSRID(ST_Point(lon, lat), 4326)"))

portfolio_df.createOrReplaceTempView("portfolio")

print(f"Portfolio: {portfolio_df.count()} policies, "
      f"TIV ${portfolio_df.agg(f.sum('tiv')).first()[0]/1e6:.1f}M")
portfolio_df.select("policy_id", "property_name", "state", "occupancy", "tiv") \
            .show(truncate=False)

## 3. Flood Exposure — FEMA Flood Zones

Point-in-polygon each location against parcels carrying FEMA flood zone
designations. Map the raw zone to a risk tier and a numeric hazard score
comparable across perils.

In [ ]:
flood_df = sedona.sql("""
    SELECT
        p.policy_id,
        COALESCE(r.fema_flood_zone, 'NONE')      AS fema_zone,
        COALESCE(r.fema_flood_zone_subtype, '')  AS fema_subtype,
        CASE
            WHEN r.fema_flood_zone IN ('VE','V')                 THEN 'Extreme'
            WHEN r.fema_flood_zone IN ('A','AE','AO','AH','A99') THEN 'High'
            WHEN r.fema_flood_zone = 'X'
                 AND r.fema_flood_zone_subtype = '0.2 PCT ANNUAL CHANCE FLOOD HAZARD'
                                                                  THEN 'Moderate'
            WHEN r.fema_flood_zone = 'X'                          THEN 'Minimal'
            ELSE 'Unmapped'
        END AS flood_tier,
        CASE
            WHEN r.fema_flood_zone IN ('VE','V')                  THEN 1.00
            WHEN r.fema_flood_zone IN ('A','AE','AO','AH','A99')  THEN 0.70
            WHEN r.fema_flood_zone = 'X'
                 AND r.fema_flood_zone_subtype = '0.2 PCT ANNUAL CHANCE FLOOD HAZARD'
                                                                  THEN 0.30
            WHEN r.fema_flood_zone = 'X'                          THEN 0.05
            ELSE 0.00
        END AS flood_score
    FROM portfolio p
    LEFT JOIN wherobots_open_data.partner_samples.regrid_parcels r
      ON ST_Intersects(p.geometry, r.geometry)
""").cache()
flood_df.createOrReplaceTempView("flood_exposure")

flood_df.show(truncate=False)

## 4. Wildfire Exposure — FPA Fire Perimeters

Filter the 1992-2015 FPA fire dataset to recent, large fires in the
Western states, buffer each point by the radius implied by `FIRE_SIZE`
acres, then intersect with the portfolio. Score bucketed by the largest
fire whose footprint contains the property.

In [ ]:
wildfire_df = sedona.sql("""
    WITH fire_buffers AS (
        SELECT
            FIRE_NAME,
            FIRE_YEAR,
            STATE,
            CAST(FIRE_SIZE AS DOUBLE) AS fire_acres,
            ST_Buffer(
                geometry,
                SQRT(CAST(FIRE_SIZE AS DOUBLE) * 4046.86 / 3.14159) / 111000.0
            ) AS burn_perim
        FROM wherobots_pro_data.weather.wild_fires
        WHERE STATE IN ('CA','OR','WA','NV','ID','MT','UT','AZ','CO')
          AND CAST(FIRE_YEAR AS INT) >= 2005
          AND CAST(FIRE_SIZE AS DOUBLE) >= 5000
    )
    SELECT
        p.policy_id,
        COUNT(fb.FIRE_NAME)                        AS fire_footprint_hits,
        COALESCE(MAX(fb.fire_acres), 0)            AS max_fire_acres,
        CASE
            WHEN MAX(fb.fire_acres) >= 100000 THEN 'Extreme'
            WHEN MAX(fb.fire_acres) >=  50000 THEN 'High'
            WHEN MAX(fb.fire_acres) >=  10000 THEN 'Moderate'
            WHEN MAX(fb.fire_acres) >      0  THEN 'Low'
            ELSE 'None'
        END AS wildfire_tier,
        CASE
            WHEN MAX(fb.fire_acres) >= 100000 THEN 0.90
            WHEN MAX(fb.fire_acres) >=  50000 THEN 0.70
            WHEN MAX(fb.fire_acres) >=  10000 THEN 0.45
            WHEN MAX(fb.fire_acres) >      0  THEN 0.20
            ELSE 0.00
        END AS wildfire_score
    FROM portfolio p
    LEFT JOIN fire_buffers fb
      ON ST_Intersects(p.geometry, fb.burn_perim)
    GROUP BY p.policy_id
""").cache()
wildfire_df.createOrReplaceTempView("wildfire_exposure")

wildfire_df.orderBy(f.col("max_fire_acres").desc()).show(truncate=False)

## 5. Seismic Exposure — PGA Hazard Zones

Five broad polygons covering the principal U.S. seismic hazard regions,
each labeled with its peak ground acceleration (PGA, g) band. In
production you would ingest the full USGS National Seismic Hazard
gridded dataset; here we declare the high-hazard envelopes inline so the
demo runs with no external dependencies.

In [ ]:
seismic_zones = [
    # zone_name,             pga_g, min_lon, min_lat, max_lon, max_lat
    ("SF Bay Area",           0.55, -122.80, 37.20, -121.50, 38.30),
    ("LA Basin",              0.45, -118.70, 33.70, -117.50, 34.35),
    ("Cascadia Subduction",   0.40, -124.50, 44.00, -122.20, 49.00),
    ("New Madrid",            0.25,  -90.80, 35.00,  -89.00, 37.00),
    ("Wasatch Front",         0.30, -112.20, 40.30, -111.60, 41.60),
]

seismic_schema = StructType([
    StructField("zone_name", StringType()),
    StructField("pga_g",     DoubleType()),
    StructField("min_lon",   DoubleType()),
    StructField("min_lat",   DoubleType()),
    StructField("max_lon",   DoubleType()),
    StructField("max_lat",   DoubleType()),
])

seismic_df = sedona.createDataFrame(seismic_zones, seismic_schema) \
    .withColumn(
        "geometry",
        f.expr("ST_MakeEnvelope(min_lon, min_lat, max_lon, max_lat, 4326)")
    )
seismic_df.createOrReplaceTempView("seismic_zones")

seismic_exposure_df = sedona.sql("""
    SELECT
        p.policy_id,
        COALESCE(MAX(s.zone_name), 'Outside high-PGA zones') AS seismic_zone,
        COALESCE(MAX(s.pga_g), 0.0)                           AS pga_g,
        CASE
            WHEN MAX(s.pga_g) >= 0.50 THEN 'Extreme'
            WHEN MAX(s.pga_g) >= 0.40 THEN 'High'
            WHEN MAX(s.pga_g) >= 0.25 THEN 'Moderate'
            WHEN MAX(s.pga_g) >  0.00 THEN 'Low'
            ELSE 'Minimal'
        END AS seismic_tier,
        CASE
            WHEN MAX(s.pga_g) >= 0.50 THEN 1.00
            WHEN MAX(s.pga_g) >= 0.40 THEN 0.75
            WHEN MAX(s.pga_g) >= 0.25 THEN 0.45
            WHEN MAX(s.pga_g) >  0.00 THEN 0.20
            ELSE 0.05
        END AS seismic_score
    FROM portfolio p
    LEFT JOIN seismic_zones s
      ON ST_Intersects(p.geometry, s.geometry)
    GROUP BY p.policy_id
""").cache()
seismic_exposure_df.createOrReplaceTempView("seismic_exposure")

seismic_exposure_df.orderBy(f.col("pga_g").desc()).show(truncate=False)

## 6. Combined Per-Policy Exposure

Join the three peril layers back to the portfolio, compute `exposed_tiv`
= `tiv × hazard_score` for each peril. This is the row-level granularity
the Iceberg table will hold.

In [ ]:
exposure_df = sedona.sql("""
    WITH combined AS (
        SELECT
            p.policy_id, p.property_name, p.state, p.occupancy,
            p.tiv, p.geometry,
            fl.flood_tier,    fl.fema_zone,      fl.flood_score,
            wf.wildfire_tier, wf.max_fire_acres, wf.wildfire_score,
            se.seismic_tier,  se.seismic_zone,   se.pga_g, se.seismic_score
        FROM portfolio p
        LEFT JOIN flood_exposure    fl USING (policy_id)
        LEFT JOIN wildfire_exposure wf USING (policy_id)
        LEFT JOIN seismic_exposure  se USING (policy_id)
    )
    SELECT policy_id, property_name, state, occupancy, tiv, geometry,
           'flood'    AS peril, flood_tier    AS hazard_tier,
           flood_score AS hazard_score,
           ROUND(tiv * flood_score, 2) AS exposed_tiv
    FROM combined
    UNION ALL
    SELECT policy_id, property_name, state, occupancy, tiv, geometry,
           'wildfire' AS peril, wildfire_tier AS hazard_tier,
           wildfire_score AS hazard_score,
           ROUND(tiv * wildfire_score, 2) AS exposed_tiv
    FROM combined
    UNION ALL
    SELECT policy_id, property_name, state, occupancy, tiv, geometry,
           'seismic'  AS peril, seismic_tier  AS hazard_tier,
           seismic_score AS hazard_score,
           ROUND(tiv * seismic_score, 2) AS exposed_tiv
    FROM combined
""").cache()
exposure_df.createOrReplaceTempView("portfolio_exposure_rows")

print(f"Per-(policy, peril) rows: {exposure_df.count()}")
exposure_df.select("policy_id", "state", "peril", "hazard_tier",
                   "hazard_score", "exposed_tiv") \
           .orderBy(f.col("exposed_tiv").desc()) \
           .show(15, truncate=False)

## 7. Aggregate by Peril × Region

The rollup underwriting typically wants: one number per peril per state.

In [ ]:
rollup_df = sedona.sql("""
    SELECT
        peril,
        state,
        COUNT(DISTINCT policy_id)                AS policy_count,
        ROUND(SUM(tiv), 0)                       AS total_tiv,
        ROUND(SUM(exposed_tiv), 0)               AS total_exposed_tiv,
        ROUND(SUM(exposed_tiv) / SUM(tiv), 4)    AS exposure_ratio
    FROM portfolio_exposure_rows
    GROUP BY peril, state
    ORDER BY peril, total_exposed_tiv DESC
""")
rollup_df.show(truncate=False)

portfolio_total_df = sedona.sql("""
    SELECT
        peril,
        ROUND(SUM(exposed_tiv), 0)               AS peril_exposed_tiv,
        ROUND(SUM(exposed_tiv) / SUM(tiv), 4)    AS peril_exposure_ratio
    FROM portfolio_exposure_rows
    GROUP BY peril
    ORDER BY peril_exposed_tiv DESC
""")
portfolio_total_df.show(truncate=False)

## 8. Persist to Iceberg

`CREATE DATABASE IF NOT EXISTS` + `CREATE OR REPLACE TABLE ... AS SELECT`
writes the per-(policy, peril) DataFrame as a managed Iceberg table in
`org_catalog`. Downstream tools (SQL BI, reinsurance-submission
generators, accumulation queries) can now read directly from the
catalog.

In [ ]:
sedona.sql(f"CREATE DATABASE IF NOT EXISTS org_catalog.{TARGET_DATABASE}")

sedona.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_FQN} AS
    SELECT
        policy_id,
        property_name,
        state,
        occupancy,
        tiv,
        peril,
        hazard_tier,
        hazard_score,
        exposed_tiv,
        geometry
    FROM portfolio_exposure_rows
""")

print(f"Wrote table: {TARGET_FQN}")
sedona.sql(f"DESCRIBE TABLE {TARGET_FQN}").show(truncate=False)
sedona.sql(f"SELECT COUNT(*) AS rows, COUNT(DISTINCT policy_id) AS policies, "
           f"COUNT(DISTINCT peril) AS perils FROM {TARGET_FQN}").show()

## 9. Validation — Read Back from the Iceberg Table

Confirm the table is queryable and the top accumulations match the
in-memory rollup.

In [ ]:
sedona.sql(f"""
    SELECT peril, state,
           COUNT(*)                        AS policy_count,
           ROUND(SUM(exposed_tiv), 0)      AS total_exposed_tiv
    FROM {TARGET_FQN}
    GROUP BY peril, state
    ORDER BY total_exposed_tiv DESC
""").show(truncate=False)

sedona.sql(f"""
    SELECT policy_id, property_name, state, peril, hazard_tier, exposed_tiv
    FROM {TARGET_FQN}
    WHERE exposed_tiv > 0
    ORDER BY exposed_tiv DESC
    LIMIT 10
""").show(truncate=False)

## 10. Preview on a Map

In [ ]:
viz = SedonaKepler.create_map()
SedonaKepler.add_df(viz, seismic_df.select("zone_name", "pga_g", "geometry"),
                    name="Seismic Zones")
SedonaKepler.add_df(
    viz,
    portfolio_df.select("policy_id", "property_name", "state",
                        "occupancy", "tiv", "geometry"),
    name="Portfolio"
)
viz